# A01 — Fresnel interface

## Mathematical and physical model

Use time dependence $e^{-i\omega t}$, vacuum wavenumber $k_0=2\pi/\lambda_0$ and conserved transverse wavevector $k_\parallel$. For each isotropic, nonmagnetic layer set
$$
u_j=k_{z,j}/k_0=\sqrt{\epsilon_j-(k_\parallel/k_0)^2},\qquad
Y_{s,j}=u_j,\qquad Y_{p,j}=\epsilon_j/u_j .
$$
Choose the forward branch with $\operatorname{Im}u_j\ge 0$ for passive layers. The admittances apply to **tangential** electric-field coefficients. The notebook compares the solver outputs only after converting them to the common ray-based polarization and power conventions.

At a single interface from medium $0$ to $1$,
$$
r=\frac{Y_0-Y_1}{Y_0+Y_1},\qquad
t=\frac{2Y_0}{Y_0+Y_1},\qquad
R=|r|^2,\quad T=\frac{\operatorname{Re}Y_1}{\operatorname{Re}Y_0}|t|^2 .
$$
Selecting $Y_p$ gives the Brewster zero. In reverse incidence the transmitted power is zero beyond the critical angle while the reflected field can still change phase.

## Physical scenario

A plane wave meets a single, flat air–glass boundary. The angular sweep runs in both directions. Air-to-glass incidence shows the Brewster minimum for p polarization, while glass-to-air incidence crosses into total internal reflection. The reflected phase above the critical angle is part of the comparison.

## Possible applications

Useful as a reference for prism coupling, total-internal-reflection illumination, and polarization at glass windows. The Fresnel solution also gives a direct check on amplitude normalization and forward/reverse incidence conventions.

## Benchmark objective

Compare both incidence directions against Fresnel coefficients, including Brewster reflection and total internal reflection.

## Reproduce

Use the project environment and select **Run All**. The notebook loads each solver's last tested settings from the published convergence study and reruns the points with paired convergence evidence. The native [Lumerical model](model.fsp) and [reconstruction script](model.lsf) illustrate the geometry for point 0 of [case.yaml](case.yaml).

[Published comparisons and figures](../../../BENCHMARK_REPORT.md#a01)

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

from rcwa_cv.case_loader import catalog
from rcwa_cv.notebook import case_setup, show_results, show_settings, show_versions, solve_sweep
from rcwa_cv.reporting.plots import geometry_figure

case_path = catalog("A01")[0]
spec, benchmark = case_setup(case_path)
results = {}
show_settings(benchmark)

figure = geometry_figure(case_path)
display(figure)
plt.close(figure)

## Analytic multilayer reference

In [ ]:
from rcwa_cv.case_loader import resolve
from rcwa_cv.models import SolverSettings
from rcwa_cv.references.multilayer import solve as solve_reference

results["reference"] = {}
for i, row in enumerate(benchmark["solvers"]["lumerical"]):
    if row["full_converged"]:
        settings = SolverSettings(**row["sample"]["settings"])
        case = resolve(spec, row["variant"], row["point"], settings)
        results["reference"][i] = solve_reference(case, settings)

## meent · NumPy CPU

In [ ]:
from rcwa_cv.solvers.meent_cpu import probe

if benchmark["comparisons"]["meent"]:
    print(probe())
results["meent"] = solve_sweep("meent", spec, benchmark)

## torcwa · CUDA GPU, complex128

In [ ]:
from rcwa_cv.solvers.torcwa_cuda import probe

if benchmark["comparisons"]["torcwa"]:
    print(probe())
results["torcwa"] = solve_sweep("torcwa", spec, benchmark)

## grcwa · NumPy CPU

In [ ]:
from rcwa_cv.solvers.grcwa_cpu import probe

if benchmark["comparisons"]["grcwa"]:
    print(probe())
results["grcwa"] = solve_sweep("grcwa", spec, benchmark)

## Lumerical RCWA

The cell uses the licensed FDTD application through its Python API. It reuses one session and applies the numerical mode and settings recorded for each eligible point.

In [ ]:
from rcwa_cv.solvers.lumerical import probe

if any(benchmark["comparisons"].values()):
    print(probe())
results["lumerical"] = solve_sweep("lumerical", spec, benchmark)

## Compare with Lumerical

The table and plots compare each open-source solver with Lumerical only at points where both have convergence evidence. Power-only pairs appear only in the power comparison. Point indices match the settings table above.

In [ ]:
show_versions(benchmark)
show_results(benchmark, results)